<a href="https://colab.research.google.com/github/AlphinDThomas/MINOR-CLA-PROJECT/blob/main/Fine%20Tuned%20Distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers accelerate scikit-learn
!python traindistilbert.py

python3: can't open file '/content/traindistilbert.py': [Errno 2] No such file or directory


In [2]:
%%writefile traindistilbert.py
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, precision_recall_curve, auc
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# 1. Load dataset files created in Task 0
train_val_df = pd.read_csv("train_val_reviews.csv")

# Standardize text column name
if "text_" in train_val_df.columns and "text" not in train_val_df.columns:
    train_val_df = train_val_df.rename(columns={"text_": "text"})

# Train on folds 1-4, evaluate on fold 0
train_df = train_val_df[train_val_df["fold"] != 0].copy()
val_df = train_val_df[train_val_df["fold"] == 0].copy()

print(f"Training samples: {len(train_df)} | Validation samples (Fold 0): {len(val_df)}")

# 2. Tokenization setup (max_length=128)
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[["text", "target"]].rename(columns={"target": "label"}))
val_ds = Dataset.from_pandas(val_df[["text", "target"]].rename(columns={"target": "label"}))

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

# 3. Load pre-trained DistilBERT sequence classifier
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Metrics computation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)

    precision_curve, recall_curve, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(recall_curve, precision_curve)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
    }

# 5. Training Arguments (Task 1 Specs)
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoint",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=use_fp16,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none"
)

# 6. Trainer initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 7. Start fine-tuning
print("\nStarting DistilBERT fine-tuning...")
trainer.train()

# 8. Final evaluation on Fold 0
metrics = trainer.evaluate()
print("\n" + "="*50)
print("         TASK 1 - FOLD 0 EVALUATION RESULTS")
print("="*50)
print(f"Accuracy  : {metrics['eval_accuracy'] * 100:.2f}%")
print(f"Precision : {metrics['eval_precision'] * 100:.2f}%")
print(f"Recall    : {metrics['eval_recall'] * 100:.2f}%")
print(f"F1 Score  : {metrics['eval_f1'] * 100:.2f}%")
print(f"PR AUC    : {metrics['eval_pr_auc']:.4f}")
print("="*50)

# Save final checkpoint locally
trainer.save_model("./distilbert_checkpoint")
tokenizer.save_pretrained("./distilbert_checkpoint")
print("\nSaved best checkpoint to ./distilbert_checkpoint")

Writing traindistilbert.py


In [3]:
!python traindistilbert.py

Training samples: 29102 | Validation samples (Fold 0): 7276
config.json: 100% 483/483 [00:00<00:00, 2.19MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 267kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 16.5MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 4.54MB/s]
Map: 100% 29102/29102 [00:06<00:00, 4167.81 examples/s]
Map: 100% 7276/7276 [00:01<00:00, 4019.00 examples/s]

model.safetensors: downloading bytes:  86% 232M/268M [00:01<00:00, 247MB/s, 19.1MB/s  ]
model.safetensors: reconstructing file:  26% 68.6M/268M [00:01<00:04, 43.0MB/s]
model.safetensors: downloading bytes: 100% 250M/250M [00:02<00:00, 118MB/s, 22.7MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:02<00:00, 126MB/s, 25.1MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 6714.97it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED 

In [4]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, average_precision_score, classification_report, confusion_matrix

MODEL_PATH = "./distilbert_checkpoint"
TEST_FILE = "heldout_reviews.csv"

# 1. Load held-out test set
df_heldout = pd.read_csv(TEST_FILE)

# Auto-detect column names from Task 0
text_col = "text_" if "text_" in df_heldout.columns else ("text" if "text" in df_heldout.columns else "review")
label_col = "target" if "target" in df_heldout.columns else "label"

# 2. Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# 3. Tokenize inputs
encodings = tokenizer(
    df_heldout[text_col].astype(str).tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(encodings, df_heldout[label_col].values)

# 4. Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    pr_auc = average_precision_score(labels, probs)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
    }

# 5. Run evaluation pass
trainer = Trainer(model=model, compute_metrics=compute_metrics)
raw_results = trainer.evaluate(test_dataset)

# Extract detailed predictions
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = df_heldout[label_col].values

# Display results
print("\n" + "=" * 52)
print("       HELDOUT TEST SET EVALUATION RESULTS")
print("=" * 52)
print(f"Accuracy  : {raw_results['eval_accuracy'] * 100:.2f}%")
print(f"Precision : {raw_results['eval_precision'] * 100:.2f}%")
print(f"Recall    : {raw_results['eval_recall'] * 100:.2f}%")
print(f"F1 Score  : {raw_results['eval_f1'] * 100:.2f}%")
print(f"PR AUC    : {raw_results['eval_pr_auc']:.4f}")
print("=" * 52)

print("\n--- Detailed Classification Report ---")
print(classification_report(true_labels, preds, target_names=["OR / Genuine (0)", "CG / Fake (1)"]))

cm = confusion_matrix(true_labels, preds)
print("--- Confusion Matrix ---")
print(f"True Negatives  (Genuine correctly flagged): {cm[0][0]}")
print(f"False Positives (Genuine misclassified)     : {cm[0][1]}")
print(f"False Negatives (Fake review missed)        : {cm[1][0]}")
print(f"True Positives  (Fake review caught)        : {cm[1][1]}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1,Pr Auc
No log,0.086353,0,0.975507,0.961557,0.990594,0.975860,0.998483



       HELDOUT TEST SET EVALUATION RESULTS
Accuracy  : 97.55%
Precision : 96.16%
Recall    : 99.06%
F1 Score  : 97.59%
PR AUC    : 0.9985

--- Detailed Classification Report ---
                  precision    recall  f1-score   support

OR / Genuine (0)       0.99      0.96      0.98      2022
   CG / Fake (1)       0.96      0.99      0.98      2020

        accuracy                           0.98      4042
       macro avg       0.98      0.98      0.98      4042
    weighted avg       0.98      0.98      0.98      4042

--- Confusion Matrix ---
True Negatives  (Genuine correctly flagged): 1942
False Positives (Genuine misclassified)     : 80
False Negatives (Fake review missed)        : 19
True Positives  (Fake review caught)        : 2001


In [7]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

MODEL_PATH = "./distilbert_checkpoint"
CUSTOM_CSV = "test_dataset.csv"  # Rename to match your file name

df = pd.read_csv(CUSTOM_CSV)

# Auto-detect text and label columns
text_col = [c for c in df.columns if c in ["review", "text", "text_"]][0]
label_col = [c for c in df.columns if c in ["label", "target"]][0]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

encodings = tokenizer(
    df[text_col].astype(str).tolist(),
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors="pt"
)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(encodings, df[label_col].values)
trainer = Trainer(model=model)

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = df[label_col].values

print("\n" + "=" * 52)
print("     CUSTOM 50/50 DATASET EVALUATION RESULTS")
print("=" * 52)
print(classification_report(true_labels, preds, target_names=["OR (Genuine)", "CG (Fake)"]))

cm = confusion_matrix(true_labels, preds)
print("--- Confusion Matrix ---")
print(f"True Negatives  (Genuine correctly flagged): {cm[0][0]}")
print(f"False Positives (Genuine misclassified)     : {cm[0][1]}")
print(f"False Negatives (Fake review missed)        : {cm[1][0]}")
print(f"True Positives  (Fake review caught)        : {cm[1][1]}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


     CUSTOM 50/50 DATASET EVALUATION RESULTS
              precision    recall  f1-score   support

OR (Genuine)       0.49      0.96      0.65        50
   CG (Fake)       0.33      0.02      0.04        50

    accuracy                           0.49       100
   macro avg       0.41      0.49      0.35       100
weighted avg       0.41      0.49      0.35       100

--- Confusion Matrix ---
True Negatives  (Genuine correctly flagged): 48
False Positives (Genuine misclassified)     : 2
False Negatives (Fake review missed)        : 49
True Positives  (Fake review caught)        : 1


In [8]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./distilbert_checkpoint"

# 1. Load the 20 custom CG reviews
cg_reviews = [
    "The product was absolutely amazing and exceeded my expectations in every possible way. The quality was excellent, the design was impressive, and the overall experience was truly wonderful. I would highly recommend this product to anyone looking for something reliable, efficient, and high-quality.",
    "I recently purchased this product, and I must say that I am extremely satisfied with my decision. From the moment I received it, everything was perfect. The product is well-designed, easy to use, and offers excellent value for money. Overall, this was a fantastic purchase.",
    "This restaurant provided an exceptional dining experience. The food was delicious, the staff were friendly and professional, and the atmosphere was pleasant and welcoming. Every aspect of the experience was carefully executed. I would definitely recommend this restaurant to friends, family, and anyone seeking a memorable meal.",
    "I was thoroughly impressed with the quality and performance of this product. It combines functionality, convenience, and affordability in an excellent manner. The user-friendly design makes it suitable for a wide range of people. Overall, it is a great product that successfully delivers on its promises.",
    "This hotel exceeded my expectations and provided an unforgettable stay. The rooms were clean and comfortable, the staff were extremely helpful, and the facilities were excellent. The location was also convenient, making it easy to explore the surrounding area. I would happily stay here again in the future.",
    "My experience with this service was outstanding from beginning to end. The process was smooth, efficient, and convenient. The staff demonstrated professionalism and genuine commitment to customer satisfaction. I appreciate the attention to detail and would confidently recommend this service to others.",
    "This product is truly a remarkable combination of quality, innovation, and practicality. I was particularly impressed by how easy it was to use and how effectively it performed its intended function. It offers excellent value and represents a worthwhile investment for anyone considering purchasing it.",
    "I had a wonderful experience using this application. The interface is clean, intuitive, and user-friendly, while the features are thoughtfully designed to meet the needs of users. Everything worked smoothly without unnecessary complications. Overall, this is an excellent application that I would strongly recommend.",
    "The customer service I received was exceptional. My concerns were addressed promptly, professionally, and respectfully. The representative demonstrated patience and understanding throughout the entire interaction. It is refreshing to encounter a company that genuinely prioritizes customer satisfaction. I am extremely pleased with the outcome.",
    "This book was an inspiring and engaging read that provided valuable insights from beginning to end. The writing was clear, informative, and thought-provoking. Each chapter contributed meaningfully to the overall message. I would recommend this book to anyone interested in expanding their knowledge and perspective.",
    "I am extremely pleased with this purchase. The product arrived in excellent condition and performed exactly as expected. Its high-quality construction, attractive appearance, and reliable performance make it an excellent choice. Considering all of its benefits, I believe it provides exceptional value for the price.",
    "This café offers a truly delightful experience for customers. The food and beverages were fresh, flavorful, and beautifully presented. The atmosphere was relaxing, and the staff were polite and attentive. Every detail contributed to a positive experience. I would certainly recommend visiting this café.",
    "I highly appreciate the effort and dedication demonstrated by this company. The product reflects careful planning, thoughtful design, and attention to customer needs. It is both practical and aesthetically pleasing. Overall, my experience was extremely positive, and I would confidently recommend this product to others.",
    "This was an excellent experience that exceeded my expectations in multiple ways. The quality was impressive, the service was efficient, and the overall process was extremely convenient. I especially appreciated the professionalism and attention to detail. Without hesitation, I would recommend this to anyone looking for a dependable option.",
    "The product offers an impressive balance between affordability, quality, and performance. It is simple to operate while still providing all the essential features required by users. The overall design is thoughtful and practical. In conclusion, this is a highly recommended product that provides excellent value.",
    "I had a very positive experience with this company. From the initial interaction to the final result, everything was handled efficiently and professionally. The communication was clear, the service was reliable, and my expectations were successfully met. I am completely satisfied and would gladly use this service again.",
    "This destination is truly beautiful and offers something for everyone. The surroundings are impressive, the atmosphere is peaceful, and the overall experience is memorable. Whether you are traveling alone, with friends, or with family, this is a wonderful place to visit. I would definitely recommend it.",
    "I was extremely impressed by the overall quality of this experience. Everything was organized carefully and delivered with professionalism. The attention to detail was particularly noticeable and contributed significantly to my satisfaction. Overall, this is an excellent choice for anyone seeking quality, reliability, and convenience.",
    "This product successfully combines modern design, practical functionality, and dependable performance. I found it easy to understand and convenient to use. The quality is clearly evident in every aspect of the product. Based on my experience, I would highly recommend it to anyone considering a similar purchase.",
    "Overall, this was an exceptional experience that left a very positive impression. The quality, service, convenience, and attention to detail were all excellent. Everything worked exactly as expected, and I was completely satisfied with the result. I would definitely recommend this product or service to others."
]

# 2. Load model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

# 3. Tokenize & predict
inputs = tokenizer(cg_reviews, padding=True, truncation=True, max_length=512, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    cg_probs = probs[:, 1].cpu().numpy()  # Probability of Class 1 (CG)
    preds = np.argmax(outputs.logits.cpu().numpy(), axis=-1)

# 4. Generate scores
total_reviews = len(cg_reviews)
caught_cg = np.sum(preds == 1)
cg_detection_score = (caught_cg / total_reviews) * 100
avg_confidence = np.mean(cg_probs) * 100

print("\n" + "=" * 55)
print("        20 CUSTOM CG REVIEWS TEST RESULTS")
print("=" * 55)
print(f"Total CG Reviews Tested  : {total_reviews}")
print(f"Correctly Caught as CG  : {caught_cg} / {total_reviews}")
print(f"CG Detection Rate Score  : {cg_detection_score:.2f}%")
print(f"Average CG Confidence    : {avg_confidence:.2f}%")
print("=" * 55)

# Detailed table breakdown
df_results = pd.DataFrame({
    "Review #": range(1, total_reviews + 1),
    "Prediction": ["CG (Fake)" if p == 1 else "OR (Genuine)" for p in preds],
    "CG Confidence (%)": np.round(cg_probs * 100, 2),
    "Status": ["✅ Caught" if p == 1 else "❌ Missed" for p in preds]
})

print("\n--- Individual Review Predictions ---")
print(df_results.to_string(index=False))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


        20 CUSTOM CG REVIEWS TEST RESULTS
Total CG Reviews Tested  : 20
Correctly Caught as CG  : 5 / 20
CG Detection Rate Score  : 25.00%
Average CG Confidence    : 25.99%

--- Individual Review Predictions ---
 Review #   Prediction  CG Confidence (%)   Status
        1    CG (Fake)          99.949997 ✅ Caught
        2    CG (Fake)          99.389999 ✅ Caught
        3 OR (Genuine)          26.020000 ❌ Missed
        4    CG (Fake)          89.730003 ✅ Caught
        5    CG (Fake)          99.790001 ✅ Caught
        6 OR (Genuine)           0.050000 ❌ Missed
        7 OR (Genuine)           0.160000 ❌ Missed
        8 OR (Genuine)           0.080000 ❌ Missed
        9 OR (Genuine)           0.040000 ❌ Missed
       10 OR (Genuine)           0.060000 ❌ Missed
       11 OR (Genuine)           0.040000 ❌ Missed
       12 OR (Genuine)           0.210000 ❌ Missed
       13 OR (Genuine)           0.050000 ❌ Missed
       14 OR (Genuine)           3.720000 ❌ Missed
       15 OR (Genuine)

In [10]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./distilbert_checkpoint"

# 20 High-Confidence Pattern CG Reviews
high_confidence_cg_reviews = [
    "The item was absolutely phenomenal and exceeded my expectations in every single detail. The overall quality was superb, the setup was effortless, and the performance was genuinely outstanding. I would strongly recommend this product to anyone seeking high quality, reliability, and excellent overall value.",
    "I am beyond delighted with this purchase, and it has completely exceeded my expectations. From the moment it arrived, everything worked seamlessly. It is exceptionally well-crafted, easy to operate, and provides incredible value for money. Overall, this was a truly marvelous purchase.",
    "This gadget exceeded my highest expectations and provided a flawless user experience. The build construction is remarkably solid, the functionality is top-tier, and the results are consistently impressive. I will happily recommend this item to all my friends and family.",
    "I was thoroughly impressed by the incredible design and superior functionality of this product. It perfectly balances utility, durability, and convenience in an outstanding manner. The thoughtful engineering makes it an absolute pleasure to use. Overall, it is a magnificent product that easily delivers on every promise.",
    "This purchase exceeded my expectations and delivered an unforgettable overall experience. The materials are sturdy and premium, the operation is smooth, and the value is completely unbeatable. I would gladly purchase this item again without a single hesitation.",
    "My experience with this product was outstanding from start to finish. The craftsmanship is top-notch, the delivery was fast, and the usability is remarkably simple. I truly appreciate the high standard of quality and would confidently recommend this product to everyone.",
    "This item represents a truly remarkable combination of modern design, durability, and practical utility. I was especially amazed by how effortlessly it performed and how reliable it remains during daily use. It offers exceptional value and is definitely worth every penny.",
    "I had a wonderful experience using this product right out of the box. The overall construction feels durable and premium, while the features are intelligently structured to satisfy every user need. Everything functioned flawlessly without any issues. Overall, this is a top-quality item that I highly recommend.",
    "The quality of this item was truly exceptional and completely surpassed my expectations. Every feature was executed with extreme precision, attention to detail, and care. It is wonderfully refreshing to receive a product that lives up to its claims. I am extremely pleased with this purchase.",
    "This product was a fantastic purchase that delivered exceptional results from day one. The design is sleek, the build is solid, and the overall operation is extremely efficient. Every aspect contributes meaningfully to a great user experience. I would recommend this to anyone looking to upgrade.",
    "I am extremely satisfied with this purchase as it performed exactly as advertised. The high-grade construction, appealing aesthetic, and smooth performance make it an exemplary choice. Taking all its benefits into account, it offers unbeatable value for the price.",
    "This product offers a truly delightful experience for anyone looking for reliability and quality. The performance is smooth, the design is sleek, and the functionality is unmatched. Every single detail reflects impressive craftsmanship. I would certainly recommend purchasing this without delay.",
    "I highly admire the outstanding quality and design demonstrated by this product. It showcases meticulous planning, sturdy materials, and a deep understanding of user preferences. It is both highly functional and aesthetically pleasing. Overall, my experience was superb and I highly recommend it.",
    "This was an incredible purchase that exceeded my expectations in every possible metric. The build was impressive, the performance was efficient, and the entire setup was remarkably quick. Without any hesitation, I would recommend this product to anyone searching for a dependable option.",
    "The product strikes an impressive balance between affordability, high-end materials, and flawless performance. It is straightforward to use while providing all the essential functionality needed. The general design is both clever and practical. In conclusion, this is an outstanding item that offers great value.",
    "I had a very positive experience with this product after testing it thoroughly. From unboxing to daily operation, everything functioned smoothly and dependably. The build quality is excellent, the design is sleek, and my expectations were completely satisfied. I would gladly buy this again.",
    "This product is truly magnificent and offers an ideal solution for daily use. The construction is solid, the performance is flawless, and the general experience is deeply satisfying. Whether for personal use or as a gift, it is a wonderful choice. I definitely recommend it.",
    "I was extremely impressed by the overall execution and build quality of this item. Everything was manufactured carefully and delivered with high attention to detail. This focus on quality contributed significantly to my complete satisfaction. Overall, it is an excellent choice for anyone seeking performance and reliability.",
    "This product seamlessly unites sleek modern design, high utility, and dependable long-term performance. I found it simple to operate and immensely practical for daily tasks. The premium build is evident in every feature. Based on my experience, I strongly recommend it.",
    "Overall, this was an exceptional purchase that left a very lasting positive impression. The materials, functionality, usability, and design quality were all stellar. Everything performed precisely as expected, and I am completely satisfied. I would definitely recommend this product to others."
]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

# Run inference
inputs = tokenizer(high_confidence_cg_reviews, padding=True, truncation=True, max_length=512, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    cg_probs = probs[:, 1].cpu().numpy()  # Class 1 = CG (Fake)
    preds = np.argmax(outputs.logits.cpu().numpy(), axis=-1)

# Compute metrics
total_reviews = len(high_confidence_cg_reviews)
caught_cg = np.sum(preds == 1)
detection_rate = (caught_cg / total_reviews) * 100
avg_confidence = np.mean(cg_probs) * 100

print("\n" + "=" * 60)
print("     HIGH-CONFIDENCE CG PATTERN TEST RESULTS")
print("=" * 60)
print(f"Total CG Reviews Tested  : {total_reviews}")
print(f"Correctly Flagged as CG : {caught_cg} / {total_reviews}")
print(f"CG Detection Rate Score : {detection_rate:.2f}%")
print(f"Average CG Confidence   : {avg_confidence:.2f}%")
print("=" * 60)

# Build results table
df_results = pd.DataFrame({
    "Review #": range(1, total_reviews + 1),
    "Prediction": ["CG (Fake)" if p == 1 else "OR (Genuine)" for p in preds],
    "CG Confidence (%)": np.round(cg_probs * 100, 2),
    "Status": ["✅ Caught" if p == 1 else "❌ Missed" for p in preds]
})

print("\n--- Individual Review Predictions ---")
print(df_results.to_string(index=False))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


     HIGH-CONFIDENCE CG PATTERN TEST RESULTS
Total CG Reviews Tested  : 20
Correctly Flagged as CG : 0 / 20
CG Detection Rate Score : 0.00%
Average CG Confidence   : 2.31%

--- Individual Review Predictions ---
 Review #   Prediction  CG Confidence (%)   Status
        1 OR (Genuine)              20.48 ❌ Missed
        2 OR (Genuine)               0.04 ❌ Missed
        3 OR (Genuine)               0.12 ❌ Missed
        4 OR (Genuine)               0.06 ❌ Missed
        5 OR (Genuine)              16.52 ❌ Missed
        6 OR (Genuine)               0.23 ❌ Missed
        7 OR (Genuine)               0.04 ❌ Missed
        8 OR (Genuine)               0.03 ❌ Missed
        9 OR (Genuine)               0.04 ❌ Missed
       10 OR (Genuine)               0.04 ❌ Missed
       11 OR (Genuine)               0.04 ❌ Missed
       12 OR (Genuine)               0.05 ❌ Missed
       13 OR (Genuine)               0.17 ❌ Missed
       14 OR (Genuine)               0.63 ❌ Missed
       15 OR (Genuine) 